Component ablation A5

In [1]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

acoustic_dataset_1 = pd.read_csv('/content/drive/MyDrive/Acoustic-subject-level-dataset.csv')
acoustic_dataset = acoustic_dataset_1.drop(columns=["Participant_ID"])
print(acoustic_dataset.shape)
print(acoustic_dataset.head())

(193, 70)
   MDVP:Fo(Hz)  MDVP:Fhi(Hz)  MDVP:Flo(Hz)  MDVP:Jitter(%)  MDVP:Jitter(Abs)  \
0   118.719333    141.312833    106.029833        0.010085          0.000087   
1   113.014000    154.201333    105.652667        0.008043          0.000072   
2   223.657833    239.034500    161.921500        0.002585          0.000010   
3   141.336667    165.070500     66.233833        0.009008          0.000065   
4   112.839667    208.860000    107.392833        0.004850          0.000042   

   MDVP:RAP  MDVP:PPQ  Jitter:DDP  MDVP:Shimmer  MDVP:Shimmer(dB)  ...  \
0  0.004998  0.007312    0.014992      0.053932          0.515167  ...   
1  0.003508  0.005035    0.010527      0.029677          0.288667  ...   
2  0.001463  0.001425    0.004393      0.017093          0.151333  ...   
3  0.005090  0.004368    0.015263      0.042640          0.407333  ...   
4  0.002468  0.002817    0.007405      0.025015          0.224500  ...   

   Delta4  Delta5  Delta6  Delta7  Delta8  Delta9  Delta10  Delt

In [3]:
import pandas as pd

meg_dataset = pd.read_csv('/content/drive/MyDrive/final_meg_rest_21_par-lev_final.csv')

print("Loaded data shape:", meg_dataset.shape)
print(meg_dataset.head())

Loaded data shape: (127, 106)
   burst_rate_L_mean  burst_rate_L_std  burst_rate_R_mean  burst_rate_R_std  \
0           4.412365          0.242544           4.486517          0.188241   
1           3.828010          0.316999           3.783657          0.297773   
2           4.645359          0.130894           4.713304          0.111269   
3           4.142208          0.243497           4.183916          0.217601   
4           3.923527          0.336133           4.035406          0.306960   

   burst_rate_asym  mean_burst_duration_L_mean  mean_burst_duration_L_std  \
0        -0.008333                    0.056832                   0.003237   
1         0.005827                    0.065725                   0.005382   
2        -0.007260                    0.053844                   0.001551   
3        -0.005009                    0.060555                   0.003554   
4        -0.014057                    0.064178                   0.005454   

   mean_burst_duration_R_mean  m

Experiment	Residual block	DANN / GRL	SCL	Purpose
A6: MIRA-Net without residual connections	✗	✓	✓

In [4]:
# ============================================================
# MIRA-NET ABLATION A6: WITHOUT RESIDUAL CONNECTIONS
# Residual connections: OFF | DANN/GRL: ON | SCL: ON
#
# Pipeline retained:
# 1. Stratified 5-fold CV for acoustic and MEG separately
# 2. Train-fold-only numerical preprocessing
# 3. RF-SHAP Top-7 feature selection for each modality
# 4. Separate train-fold StandardScaler per modality
# 5. Shared encoder with DANN + supervised contrastive learning
# 6. 32-D latent representations
# 7. Final MLP, held-out predictions and OOF metrics
# 8. Save OOF embeddings, metrics and selected features
#
# Required variables:
# acoustic_dataset, meg_dataset
# Each must be a DataFrame with the binary label in its last column.
# Labels: 0 = HC, 1 = PD. Participant ID columns must be removed.
# ============================================================

import random
import warnings

import numpy as np
import pandas as pd
import shap

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.autograd import Function
from torch.utils.data import DataLoader, TensorDataset

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

warnings.filterwarnings("ignore", category=UserWarning)


# ============================================================
# DEVICE AND REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Device:", device)


# ============================================================
# MODEL SETTINGS
# ============================================================

LATENT_DIM = 32
TOP_K = 7
N_SPLITS = 5

RF_N_ESTIMATORS = 300
NUMERICAL_CAP = 1e12

BATCH_SIZE = 128
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
ENCODER_EPOCHS = 120

TASK_WEIGHT = 3.0
DOMAIN_WEIGHT = 0.25
SCL_WEIGHT = 0.55
SCL_TEMPERATURE = 0.07

GRL_MAX_LAMBDA = 0.25
GRL_WARMUP_EPOCHS = 50

MLP_EPOCHS = 80
MLP_LEARNING_RATE = 1e-3


# ============================================================
# LOAD DATA AND EXTRACT FEATURES/LABELS
# ============================================================

df_acoustic_full = acoustic_dataset.copy()
df_meg_full = meg_dataset.copy()


def extract_features_and_labels(df, dataset_name):

    if df.shape[1] < 2:
        raise ValueError(
            f"{dataset_name} must contain features and a label."
        )

    y = pd.to_numeric(
        df.iloc[:, -1],
        errors="raise"
    ).to_numpy(dtype=np.int64)

    X_df = df.iloc[:, :-1].apply(
        pd.to_numeric,
        errors="coerce"
    )

    X = X_df.to_numpy(dtype=np.float64)
    X[~np.isfinite(X)] = np.nan

    if not np.isin(y, [0, 1]).all():
        raise ValueError(
            f"{dataset_name}: labels must be 0 and 1."
        )

    if X.shape[1] == 0:
        raise ValueError(
            f"{dataset_name}: no features found."
        )

    counts = np.bincount(y, minlength=2)

    if len(y) < N_SPLITS or counts.min() < N_SPLITS:
        raise ValueError(
            f"{dataset_name}: each class needs at least "
            f"{N_SPLITS} samples for stratified CV."
        )

    return X, y, list(X_df.columns)


X_acoustic_raw, y_acoustic, acoustic_feature_names = (
    extract_features_and_labels(
        df_acoustic_full, "Acoustic"
    )
)

X_meg_raw, y_meg, meg_feature_names = (
    extract_features_and_labels(
        df_meg_full, "MEG"
    )
)

print("\nDataset shapes")
print("Acoustic:", X_acoustic_raw.shape)
print("MEG     :", X_meg_raw.shape)

print(
    "Acoustic labels:",
    dict(zip(*np.unique(y_acoustic, return_counts=True)))
)

print(
    "MEG labels:",
    dict(zip(*np.unique(y_meg, return_counts=True)))
)


# ============================================================
# LEAKAGE-FREE NUMERICAL PREPROCESSING
# Fit clipping bounds and medians on TRAIN ONLY
# ============================================================

def preprocess_fold_train_test(X_train, X_test):

    X_train = np.asarray(X_train, dtype=np.float64).copy()
    X_test = np.asarray(X_test, dtype=np.float64).copy()

    if (
        X_train.ndim != 2
        or X_test.ndim != 2
        or X_train.shape[1] != X_test.shape[1]
    ):
        raise ValueError(
            "Train/test features must be 2-D with matching feature counts."
        )

    X_train[~np.isfinite(X_train)] = np.nan
    X_test[~np.isfinite(X_test)] = np.nan

    X_train = np.clip(
        X_train, -NUMERICAL_CAP, NUMERICAL_CAP
    )
    X_test = np.clip(
        X_test, -NUMERICAL_CAP, NUMERICAL_CAP
    )

    n_features = X_train.shape[1]

    lower = np.zeros(n_features, dtype=np.float64)
    upper = np.zeros(n_features, dtype=np.float64)
    medians = np.zeros(n_features, dtype=np.float64)

    # Train-only 1st/99th percentile clipping bounds.
    for j in range(n_features):

        vals = X_train[:, j]
        vals = vals[np.isfinite(vals)]

        if vals.size:
            lower[j] = np.quantile(vals, 0.01)
            upper[j] = np.quantile(vals, 0.99)
        else:
            lower[j] = 0.0
            upper[j] = 0.0

    X_train = np.maximum(
        np.minimum(X_train, upper), lower
    )
    X_test = np.maximum(
        np.minimum(X_test, upper), lower
    )

    # Train-only median imputation.
    for j in range(n_features):

        vals = X_train[:, j]
        vals = vals[np.isfinite(vals)]

        medians[j] = np.median(vals) if vals.size else 0.0

    def impute(X):

        X = np.asarray(X, dtype=np.float64).copy()
        X[~np.isfinite(X)] = np.nan

        for j in range(X.shape[1]):

            missing = ~np.isfinite(X[:, j])
            X[missing, j] = medians[j]

        return X

    X_train = impute(X_train)
    X_test = impute(X_test)

    if (
        not np.isfinite(X_train).all()
        or not np.isfinite(X_test).all()
    ):
        raise ValueError(
            "Non-finite values remain after preprocessing."
        )

    return X_train, X_test, medians, lower, upper


# ============================================================
# FOLD-WISE RF-SHAP TOP-7 FEATURE SELECTION
# ============================================================

def select_top_features_rf_shap(
    X_train,
    y_train,
    feature_names,
    top_k=7,
    random_state=42,
    dataset_name="Dataset"
):

    X_train = np.asarray(X_train, dtype=np.float64)
    y_train = np.asarray(y_train, dtype=np.int64)

    if not np.isfinite(X_train).all():
        raise ValueError(
            f"{dataset_name}: non-finite values before RF fitting."
        )

    k = min(top_k, X_train.shape[1])

    if k < 1:
        raise ValueError(
            f"{dataset_name}: no features available."
        )

    rf = RandomForestClassifier(
        n_estimators=RF_N_ESTIMATORS,
        random_state=random_state,
        class_weight="balanced",
        n_jobs=-1
    )

    rf.fit(X_train, y_train)

    shap_values = shap.TreeExplainer(rf).shap_values(X_train)

    # Support list and array SHAP formats.
    if isinstance(shap_values, list):

        sv = np.asarray(
            shap_values[1 if len(shap_values) > 1 else 0]
        )

    else:

        sv = np.asarray(shap_values)

        if sv.ndim == 3:

            # Samples x features x classes.
            if (
                sv.shape[0] == len(X_train)
                and sv.shape[1] == X_train.shape[1]
            ):
                sv = sv[:, :, min(1, sv.shape[2] - 1)]

            # Classes x samples x features.
            elif (
                sv.shape[1] == len(X_train)
                and sv.shape[2] == X_train.shape[1]
            ):
                sv = sv[min(1, sv.shape[0] - 1), :, :]

            else:
                raise ValueError(
                    f"{dataset_name}: unexpected SHAP shape {sv.shape}."
                )

    if sv.shape != X_train.shape or not np.isfinite(sv).all():
        raise ValueError(
            f"{dataset_name}: invalid SHAP shape or values: {sv.shape}."
        )

    importance = np.mean(np.abs(sv), axis=0)
    selected = np.argsort(importance)[::-1][:k]
    names = [feature_names[i] for i in selected]

    print(f"\n{dataset_name} RF-SHAP Top-{k}:")

    for rank, idx in enumerate(selected, start=1):
        print(
            f"{rank:02d}. {feature_names[idx]} "
            f"(mean |SHAP|={importance[idx]:.6f})"
        )

    return selected, names


# ============================================================
# GRADIENT REVERSAL LAYER
# ============================================================

class GradReverse(Function):

    @staticmethod
    def forward(ctx, x, lambd):
        ctx.lambd = float(lambd)
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambd * grad_output, None


def grad_reverse(x, lambd):
    return GradReverse.apply(x, lambd)


# ============================================================
# A6 MODEL: PLAIN ENCODER + DANN + SCL
#
# IMPORTANT:
# The residual block and skip connection are removed.
# DANN and SCL are retained.
# ============================================================

class DANNContrastive(nn.Module):

    def __init__(self, input_dim, latent_dim=32):

        super().__init__()

        # No ResidualBlock and no skip connection.
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Dropout(0.1),

            nn.Linear(64, latent_dim)
        )

        self.task_head = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.ReLU(),

            nn.Dropout(0.2),

            nn.Linear(32, 2)
        )

        self.domain_head = nn.Sequential(
            nn.Linear(latent_dim, 16),
            nn.ReLU(),

            nn.Linear(16, 2)
        )

    def forward(self, x, lambd=1.0):

        z = self.encoder(x)

        z_norm = F.normalize(
            z, p=2, dim=1
        )

        task_logits = self.task_head(z_norm)

        domain_logits = self.domain_head(
            grad_reverse(z_norm, lambd)
        )

        return task_logits, domain_logits, z_norm


# ============================================================
# SUPERVISED CONTRASTIVE LOSS
# ============================================================

class SupConLoss(nn.Module):

    def __init__(self, temperature=0.07):

        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):

        labels = labels.contiguous().view(-1, 1)

        mask = torch.eq(
            labels, labels.T
        ).float().to(features.device)

        similarity = torch.matmul(
            features, features.T
        ) / self.temperature

        logits_max, _ = torch.max(
            similarity, dim=1, keepdim=True
        )

        logits = similarity - logits_max.detach()

        logits_mask = (
            torch.ones_like(mask)
            - torch.eye(mask.shape[0], device=features.device)
        )

        mask = mask * logits_mask

        exp_logits = torch.exp(logits) * logits_mask

        log_prob = logits - torch.log(
            exp_logits.sum(dim=1, keepdim=True) + 1e-8
        )

        positive_count = mask.sum(dim=1)

        mean_log_prob_pos = (
            (mask * log_prob).sum(dim=1)
            / (positive_count + 1e-8)
        )

        valid_anchors = positive_count > 0

        if not valid_anchors.any():
            return features.sum() * 0.0

        return -mean_log_prob_pos[valid_anchors].mean()


# ============================================================
# GRL LAMBDA SCHEDULE
# ============================================================

def get_grl_lambda(epoch):

    progress = min(
        epoch / float(GRL_WARMUP_EPOCHS), 1.0
    )

    lambd = GRL_MAX_LAMBDA * (
        2.0 / (1.0 + np.exp(-10.0 * progress)) - 1.0
    )

    return float(lambd)


# ============================================================
# TRAIN DANN + SCL ON OUTER-TRAINING PARTICIPANTS ONLY
# ============================================================

def train_dann_scl(
    X_a,
    y_a,
    X_m,
    y_m,
    fold_number
):

    if X_a.shape[1] != X_m.shape[1]:
        raise ValueError(
            "Acoustic and MEG must have equal selected feature counts."
        )

    X_all = np.vstack([X_a, X_m]).astype(np.float32)
    y_all = np.hstack([y_a, y_m]).astype(np.int64)

    # Domain 0 = acoustic; domain 1 = MEG.
    domain_all = np.hstack([
        np.zeros(len(y_a), dtype=np.int64),
        np.ones(len(y_m), dtype=np.int64)
    ])

    if not np.isfinite(X_all).all():
        raise ValueError(
            f"Non-finite DANN input in fold {fold_number}."
        )

    dataset = TensorDataset(
        torch.tensor(X_all, dtype=torch.float32),
        torch.tensor(y_all, dtype=torch.long),
        torch.tensor(domain_all, dtype=torch.long)
    )

    generator = torch.Generator()
    generator.manual_seed(SEED + fold_number)

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        drop_last=False,
        generator=generator
    )

    model = DANNContrastive(
        input_dim=X_all.shape[1],
        latent_dim=LATENT_DIM
    ).to(device)

    optimizer = optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    task_loss_fn = nn.CrossEntropyLoss()
    domain_loss_fn = nn.CrossEntropyLoss()

    contrastive_loss_fn = SupConLoss(
        temperature=SCL_TEMPERATURE
    )

    print(
        f"\nTraining A6: No Residual Connections + DANN + SCL "
        f"| fold {fold_number}"
    )

    for epoch in range(ENCODER_EPOCHS):

        model.train()

        running_loss = 0.0
        running_task_loss = 0.0
        running_domain_loss = 0.0
        running_scl_loss = 0.0

        correct = 0
        n_seen = 0

        lambd = get_grl_lambda(epoch)

        for xb, yb, db in loader:

            xb = xb.to(device)
            yb = yb.to(device)
            db = db.to(device)

            optimizer.zero_grad()

            task_logits, domain_logits, z = model(
                xb, lambd=lambd
            )

            loss_task = task_loss_fn(
                task_logits, yb
            )

            loss_domain = domain_loss_fn(
                domain_logits, db
            )

            loss_scl = contrastive_loss_fn(
                z, yb
            )

            loss = (
                TASK_WEIGHT * loss_task
                + DOMAIN_WEIGHT * loss_domain
                + SCL_WEIGHT * loss_scl
            )

            if not torch.isfinite(loss):
                raise RuntimeError(
                    f"Non-finite loss in fold {fold_number}, "
                    f"epoch {epoch + 1}."
                )

            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            running_task_loss += loss_task.item()
            running_domain_loss += loss_domain.item()
            running_scl_loss += loss_scl.item()

            correct += (
                task_logits.argmax(dim=1) == yb
            ).sum().item()

            n_seen += len(yb)

        if epoch == 0 or (epoch + 1) % 10 == 0:

            denom = max(len(loader), 1)

            print(
                f"Epoch {epoch + 1:03d}/{ENCODER_EPOCHS} "
                f"| total={running_loss / denom:.4f} "
                f"| task={running_task_loss / denom:.4f} "
                f"| domain={running_domain_loss / denom:.4f} "
                f"| SCL={running_scl_loss / denom:.4f} "
                f"| GRL={lambd:.3f} "
                f"| train_acc={correct / max(n_seen, 1):.4f}"
            )

    return model


# ============================================================
# EXTRACT LATENT FEATURES
# ============================================================

def extract_latent(model, X):

    X = np.asarray(X, dtype=np.float32)

    if not np.isfinite(X).all():
        raise ValueError(
            "Non-finite values passed to latent extraction."
        )

    model.eval()

    X_tensor = torch.tensor(
        X,
        dtype=torch.float32,
        device=device
    )

    with torch.no_grad():

        _, _, z = model(
            X_tensor, lambd=0.0
        )

    z = z.cpu().numpy().astype(np.float32)

    if not np.isfinite(z).all():
        raise RuntimeError(
            "Non-finite latent embeddings generated."
        )

    return z


# ============================================================
# FINAL MLP CLASSIFIER
# ============================================================

class FinalMLP(nn.Module):

    def __init__(self, input_dim=32):

        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),

            nn.Dropout(0.2),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 2)
        )

    def forward(self, x):
        return self.net(x)


def train_and_evaluate_mlp(
    X_train,
    y_train,
    X_test,
    y_test,
    fold_number
):

    X_train = np.asarray(X_train, dtype=np.float32)
    X_test = np.asarray(X_test, dtype=np.float32)

    y_train = np.asarray(y_train, dtype=np.int64)
    y_test = np.asarray(y_test, dtype=np.int64)

    if (
        not np.isfinite(X_train).all()
        or not np.isfinite(X_test).all()
    ):
        raise ValueError(
            "Non-finite latent values passed to MLP."
        )

    xt = torch.tensor(
        X_train, dtype=torch.float32, device=device
    )

    xv = torch.tensor(
        X_test, dtype=torch.float32, device=device
    )

    yt = torch.tensor(
        y_train, dtype=torch.long, device=device
    )

    clf = FinalMLP(
        input_dim=LATENT_DIM
    ).to(device)

    optimizer = optim.Adam(
        clf.parameters(),
        lr=MLP_LEARNING_RATE
    )

    loss_fn = nn.CrossEntropyLoss()

    # Original full-batch MLP training for 80 epochs.
    clf.train()

    for _ in range(MLP_EPOCHS):

        optimizer.zero_grad()

        logits = clf(xt)
        loss = loss_fn(logits, yt)

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite MLP loss in fold {fold_number}."
            )

        loss.backward()
        optimizer.step()

    clf.eval()

    with torch.no_grad():

        logits = clf(xv)

        probs = F.softmax(
            logits, dim=1
        )[:, 1].cpu().numpy()

        preds = logits.argmax(
            dim=1
        ).cpu().numpy()

    tn, fp, fn, tp = confusion_matrix(
        y_test, preds, labels=[0, 1]
    ).ravel()

    metrics = {
        "Accuracy": accuracy_score(y_test, preds),
        "Precision": precision_score(
            y_test, preds, zero_division=0
        ),
        "Recall": recall_score(
            y_test, preds, zero_division=0
        ),
        "Specificity": (
            tn / (tn + fp) if (tn + fp) else np.nan
        ),
        "F1": f1_score(
            y_test, preds, zero_division=0
        ),
        "AUC": (
            roc_auc_score(y_test, probs)
            if len(np.unique(y_test)) == 2
            else np.nan
        ),
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "predictions": preds,
        "probabilities": probs
    }

    print(f"\nA6 fold {fold_number} held-out metrics:")

    for metric in [
        "Accuracy", "Precision", "Recall",
        "Specificity", "F1", "AUC"
    ]:
        print(f"{metric:12s}: {metrics[metric]:.4f}")

    print(
        "Confusion matrix [[TN, FP], [FN, TP]]: "
        f"[[{tn}, {fp}], [{fn}, {tp}]]"
    )

    return metrics


# ============================================================
# OUTER STRATIFIED 5-FOLD SPLITS
# ============================================================

acoustic_splits = list(
    StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=SEED
    ).split(X_acoustic_raw, y_acoustic)
)

meg_splits = list(
    StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=SEED
    ).split(X_meg_raw, y_meg)
)


# ============================================================
# INITIALIZE OOF ARRAYS
# ============================================================

oof_pred_acoustic = np.zeros(
    len(y_acoustic), dtype=np.int64
)

oof_prob_acoustic = np.zeros(
    len(y_acoustic), dtype=np.float32
)

oof_pred_meg = np.zeros(
    len(y_meg), dtype=np.int64
)

oof_prob_meg = np.zeros(
    len(y_meg), dtype=np.float32
)

oof_count_acoustic = np.zeros(
    len(y_acoustic), dtype=np.int64
)

oof_count_meg = np.zeros(
    len(y_meg), dtype=np.int64
)

fused_acoustic_32D = np.zeros(
    (len(y_acoustic), LATENT_DIM), dtype=np.float32
)

fused_meg_32D = np.zeros(
    (len(y_meg), LATENT_DIM), dtype=np.float32
)

fold_records = []
selected_features_by_fold = {}


# ============================================================
# OUTER FOLD LOOP
# ============================================================

print("\n" + "=" * 72)
print("A6: NO RESIDUAL CONNECTIONS | DANN/GRL ON | SCL ON")
print("=" * 72)

for fold in range(N_SPLITS):

    fold_no = fold + 1

    print(
        "\n" + "=" * 72
        + f"\nA6 OUTER FOLD {fold_no}/{N_SPLITS}\n"
        + "=" * 72
    )

    train_a_idx, test_a_idx = acoustic_splits[fold]
    train_m_idx, test_m_idx = meg_splits[fold]

    Xa_tr_raw = X_acoustic_raw[train_a_idx]
    Xa_te_raw = X_acoustic_raw[test_a_idx]

    ya_tr = y_acoustic[train_a_idx]
    ya_te = y_acoustic[test_a_idx]

    Xm_tr_raw = X_meg_raw[train_m_idx]
    Xm_te_raw = X_meg_raw[test_m_idx]

    ym_tr = y_meg[train_m_idx]
    ym_te = y_meg[test_m_idx]

    print(
        f"Acoustic train/test: "
        f"{len(train_a_idx)}/{len(test_a_idx)}"
    )

    print(
        f"MEG train/test     : "
        f"{len(train_m_idx)}/{len(test_m_idx)}"
    )

    # 1. Train-fold-only preprocessing.
    Xa_tr, Xa_te, *_ = preprocess_fold_train_test(
        Xa_tr_raw, Xa_te_raw
    )

    Xm_tr, Xm_te, *_ = preprocess_fold_train_test(
        Xm_tr_raw, Xm_te_raw
    )

    # 2. RF-SHAP Top-7 for acoustic training data.
    a_idx, a_names = select_top_features_rf_shap(
        Xa_tr,
        ya_tr,
        acoustic_feature_names,
        top_k=TOP_K,
        random_state=SEED + fold_no,
        dataset_name=f"Acoustic fold {fold_no}"
    )

    # 3. RF-SHAP Top-7 for MEG training data.
    m_idx, m_names = select_top_features_rf_shap(
        Xm_tr,
        ym_tr,
        meg_feature_names,
        top_k=TOP_K,
        random_state=SEED + 100 + fold_no,
        dataset_name=f"MEG fold {fold_no}"
    )

    if len(a_idx) != len(m_idx):
        raise ValueError(
            "Acoustic and MEG must have equal selected feature counts."
        )

    selected_features_by_fold[fold_no] = {
        "acoustic": a_names,
        "meg": m_names
    }

    # 4. Fit separate scalers on each modality's training data only.
    scaler_a = StandardScaler()

    Xa_tr = scaler_a.fit_transform(
        Xa_tr[:, a_idx]
    ).astype(np.float32)

    Xa_te = scaler_a.transform(
        Xa_te[:, a_idx]
    ).astype(np.float32)

    scaler_m = StandardScaler()

    Xm_tr = scaler_m.fit_transform(
        Xm_tr[:, m_idx]
    ).astype(np.float32)

    Xm_te = scaler_m.transform(
        Xm_te[:, m_idx]
    ).astype(np.float32)

    for name, arr in [
        ("Acoustic train", Xa_tr),
        ("Acoustic test", Xa_te),
        ("MEG train", Xm_tr),
        ("MEG test", Xm_te)
    ]:
        if not np.isfinite(arr).all():
            raise RuntimeError(
                f"{name} contains non-finite values in fold {fold_no}."
            )

    # 5. Train DANN + SCL. Residual connections are disabled.
    encoder = train_dann_scl(
        Xa_tr, ya_tr,
        Xm_tr, ym_tr,
        fold_no
    )

    # 6. Extract training and held-out 32-D embeddings.
    za_tr = extract_latent(encoder, Xa_tr)
    zm_tr = extract_latent(encoder, Xm_tr)

    za_te = extract_latent(encoder, Xa_te)
    zm_te = extract_latent(encoder, Xm_te)

    X_latent_tr = np.vstack([
        za_tr, zm_tr
    ]).astype(np.float32)

    y_latent_tr = np.hstack([
        ya_tr, ym_tr
    ]).astype(np.int64)

    X_latent_te = np.vstack([
        za_te, zm_te
    ]).astype(np.float32)

    y_latent_te = np.hstack([
        ya_te, ym_te
    ]).astype(np.int64)

    # 7. Train the final MLP and evaluate held-out participants.
    result = train_and_evaluate_mlp(
        X_latent_tr,
        y_latent_tr,
        X_latent_te,
        y_latent_te,
        fold_no
    )

    fold_records.append({
        "Fold": fold_no,
        **{
            k: result[k]
            for k in [
                "Accuracy", "Precision", "Recall",
                "Specificity", "F1", "AUC",
                "TN", "FP", "FN", "TP"
            ]
        }
    })

    # 8. Store OOF predictions and held-out embeddings.
    n_a = len(test_a_idx)

    oof_pred_acoustic[test_a_idx] = (
        result["predictions"][:n_a]
    )

    oof_prob_acoustic[test_a_idx] = (
        result["probabilities"][:n_a]
    )

    oof_pred_meg[test_m_idx] = (
        result["predictions"][n_a:]
    )

    oof_prob_meg[test_m_idx] = (
        result["probabilities"][n_a:]
    )

    fused_acoustic_32D[test_a_idx] = za_te
    fused_meg_32D[test_m_idx] = zm_te

    oof_count_acoustic[test_a_idx] += 1
    oof_count_meg[test_m_idx] += 1

    print("Held-out acoustic latent shape:", za_te.shape)
    print("Held-out MEG latent shape     :", zm_te.shape)


# ============================================================
# OOF ASSIGNMENT CHECK
# ============================================================

print("\nOOF assignment counts:")

print(
    "Acoustic:",
    np.unique(oof_count_acoustic, return_counts=True)
)

print(
    "MEG:",
    np.unique(oof_count_meg, return_counts=True)
)

assert np.all(oof_count_acoustic == 1)
assert np.all(oof_count_meg == 1)


# ============================================================
# COMBINED OOF EMBEDDINGS AND METRICS
# ============================================================

X_fused = np.vstack([
    fused_acoustic_32D,
    fused_meg_32D
]).astype(np.float32)

y_fused = np.hstack([
    y_acoustic,
    y_meg
]).astype(np.int64)

fused_oof_probs = np.hstack([
    oof_prob_acoustic,
    oof_prob_meg
]).astype(np.float32)

fused_oof_preds = np.hstack([
    oof_pred_acoustic,
    oof_pred_meg
]).astype(np.int64)

print("\n" + "=" * 72)
print("A6 COMBINED OOF RESULTS")
print("=" * 72)

print(
    f"Accuracy  : "
    f"{accuracy_score(y_fused, fused_oof_preds):.4f}"
)

print(
    f"Precision : "
    f"{precision_score(y_fused, fused_oof_preds, zero_division=0):.4f}"
)

print(
    f"Recall    : "
    f"{recall_score(y_fused, fused_oof_preds, zero_division=0):.4f}"
)

print(
    f"F1-score  : "
    f"{f1_score(y_fused, fused_oof_preds, zero_division=0):.4f}"
)

print(
    f"AUC       : "
    f"{roc_auc_score(y_fused, fused_oof_probs):.4f}"
)


# ============================================================
# MEAN AND SD ACROSS OUTER FOLDS
# ============================================================

fold_metrics_df = pd.DataFrame(fold_records)

print("\nA6 per-fold metrics:")
print(fold_metrics_df.round(4).to_string(index=False))

summary_rows = []

for metric in [
    "Accuracy", "Precision", "Recall",
    "Specificity", "F1", "AUC"
]:

    vals = fold_metrics_df[metric].to_numpy(dtype=float)

    summary_rows.append({
        "Metric": metric,
        "Mean": np.nanmean(vals),
        "SD": np.nanstd(vals, ddof=1)
    })

summary_df = pd.DataFrame(summary_rows)

print("\nA6 mean and SD across folds:")
print(summary_df.round(4).to_string(index=False))


# ============================================================
# CHECK AND SAVE 32-D OOF EMBEDDINGS
# ============================================================

assert fused_acoustic_32D.shape == (
    len(y_acoustic), LATENT_DIM
)

assert fused_meg_32D.shape == (
    len(y_meg), LATENT_DIM
)

assert X_fused.shape == (
    len(y_acoustic) + len(y_meg), LATENT_DIM
)

assert np.isfinite(X_fused).all()

latent_columns = [
    f"latent_{i+1}" for i in range(LATENT_DIM)
]

oof_acoustic_df = pd.DataFrame(
    fused_acoustic_32D,
    columns=latent_columns
)

oof_acoustic_df["Status"] = y_acoustic
oof_acoustic_df["OOF_Prediction"] = oof_pred_acoustic
oof_acoustic_df["OOF_Probability"] = oof_prob_acoustic

oof_meg_df = pd.DataFrame(
    fused_meg_32D,
    columns=latent_columns
)

oof_meg_df["Status"] = y_meg
oof_meg_df["OOF_Prediction"] = oof_pred_meg
oof_meg_df["OOF_Probability"] = oof_prob_meg

oof_acoustic_df.to_csv(
    "mira_net_A6_acoustic_oof_embeddings.csv",
    index=False
)

oof_meg_df.to_csv(
    "mira_net_A6_meg_oof_embeddings.csv",
    index=False
)

fold_metrics_df.to_csv(
    "mira_net_A6_fold_metrics.csv",
    index=False
)

summary_df.to_csv(
    "mira_net_A6_fold_summary.csv",
    index=False
)

# Save RF-SHAP selected features by fold and modality.
feature_rows = []

for fold_no, modalities in selected_features_by_fold.items():

    for modality in ["acoustic", "meg"]:

        for rank, name in enumerate(
            modalities[modality], start=1
        ):

            feature_rows.append({
                "Fold": fold_no,
                "Modality": modality,
                "Rank": rank,
                "Feature": name
            })

pd.DataFrame(feature_rows).to_csv(
    "mira_net_A6_rf_shap_selected_features.csv",
    index=False
)


# ============================================================
# COMPLETION SUMMARY
# ============================================================

print("\n" + "=" * 72)
print("A6 ABLATION COMPLETE")
print("=" * 72)

print("Residual connections: OFF")
print("DANN/GRL: ON")
print("SCL: ON")
print("Encoder epochs:", ENCODER_EPOCHS)
print("Final MLP epochs:", MLP_EPOCHS)

print("Acoustic 32-D:", fused_acoustic_32D.shape)
print("MEG 32-D     :", fused_meg_32D.shape)
print("Combined 32-D:", X_fused.shape)

print("\nSaved files:")
print("mira_net_A6_acoustic_oof_embeddings.csv")
print("mira_net_A6_meg_oof_embeddings.csv")
print("mira_net_A6_fold_metrics.csv")
print("mira_net_A6_fold_summary.csv")
print("mira_net_A6_rf_shap_selected_features.csv")

Device: cpu

Dataset shapes
Acoustic: (193, 69)
MEG     : (127, 105)
Acoustic labels: {np.int64(0): np.int64(66), np.int64(1): np.int64(127)}
MEG labels: {np.int64(0): np.int64(65), np.int64(1): np.int64(62)}

A6: NO RESIDUAL CONNECTIONS | DANN/GRL ON | SCL ON

A6 OUTER FOLD 1/5
Acoustic train/test: 154/39
MEG train/test     : 101/26

Acoustic fold 1 RF-SHAP Top-7:
01. Degree-voice breaks (mean |SHAP|=0.020099)
02. MDVP:Shimmer (mean |SHAP|=0.019161)
03. Delta1 (mean |SHAP|=0.019100)
04. RPDE (mean |SHAP|=0.018424)
05. Delta0 (mean |SHAP|=0.017792)
06. No. of voice breaks (mean |SHAP|=0.017317)
07. MFCC5 (mean |SHAP|=0.017233)

MEG fold 1 RF-SHAP Top-7:
01. burst_rate_asym (mean |SHAP|=0.019693)
02. burst_dur_cv_R_std (mean |SHAP|=0.014527)
03. burst_dur_cv_L_std (mean |SHAP|=0.013783)
04. ibi_cv_asym (mean |SHAP|=0.013600)
05. mean_burst_duration_R_std (mean |SHAP|=0.013082)
06. mean_ibi_R_std (mean |SHAP|=0.012419)
07. mean_burst_duration_asym (mean |SHAP|=0.011698)

Training A6: No 

In [5]:
import torch

# Assuming fused_acoustic_32D and fused_meg_32D are numpy arrays of shape (7350, 32)

# Step 1: Convert numpy arrays to torch tensors (float32)
fused_acoustic_tensor = torch.from_numpy(fused_acoustic_32D).float()
fused_meg_tensor = torch.from_numpy(fused_meg_32D).float()

# Step 2: Concatenate row-wise (along dim=0)
fused_output_tensor = torch.cat([fused_acoustic_tensor, fused_meg_tensor], dim=0)  # Shape: (14700, 32)

# Step 3: Print shape and dtype
print(fused_output_tensor.shape)  # torch.Size([14700, 32])
print(fused_output_tensor.dtype)  # torch.float32


torch.Size([320, 32])
torch.float32


In [6]:
import torch
import numpy as np

# ------------------------
# Step 1: Extract targets correctly
# ------------------------

# Acoustic target from df_acoustic_top7 (last column)
y_acoustic = acoustic_dataset.iloc[:, -1].values.astype(np.int32)   # (7350,)

# MEG target from final_meg_df (last column)
y_meg = meg_dataset.iloc[:, -1].values.astype(np.int32)           # (7350,)

# Convert to tensors (reshape to column)
acoustic_target_tensor = torch.tensor(y_acoustic).long().reshape(-1, 1)   # (7350, 1)
meg_target_tensor      = torch.tensor(y_meg).long().reshape(-1, 1)        # (7350, 1)

# ------------------------
# Step 2: Convert 32D latent features to tensors
# ------------------------

fused_acoustic_tensor = torch.from_numpy(fused_acoustic_32D).float()  # (7350, 32)
fused_meg_tensor      = torch.from_numpy(fused_meg_32D).float()       # (7350, 32)

# ------------------------
# Step 3: Attach target columns
# ------------------------

# Concatenate latent features with target
acoustic_full = torch.cat([fused_acoustic_tensor, acoustic_target_tensor.float()], dim=1)  # (7350, 65)
meg_full      = torch.cat([fused_meg_tensor,      meg_target_tensor.float()],      dim=1)  # (7350, 65)

# ------------------------
# Step 4: Concatenate both datasets
# ------------------------

fused_df = torch.cat([acoustic_full, meg_full], dim=0)  # (14700, 65)

# ------------------------
# Step 5: Separate features and labels for PyTorch
# ------------------------

X = fused_df[:, :-1].float()   # Features (32D) -> float32
y = fused_df[:, -1].long()     # Target -> int32

# ------------------------
# Step 6: Print shapes and dtypes
# ------------------------

print(f"Features shape: {X.shape}, dtype: {X.dtype}")  # (14700, 32), float32
print(f"Labels shape: {y.shape}, dtype: {y.dtype}")    # (14700,), int32

Features shape: torch.Size([320, 32]), dtype: torch.float32
Labels shape: torch.Size([320]), dtype: torch.int64


In [7]:
import torch
import pandas as pd

# fused_df shape = (14700, 33)

# Step 6: Convert fused tensor to numpy
fused_df_np = fused_df.numpy()

# Step 7: Create correct column names for 32 latent features + 1 label
feature_names = [f"feat_{i}" for i in range(32)]
feature_names.append("PD_Label_Combined")

# Step 8: Convert to DataFrame
fused_df_named = pd.DataFrame(fused_df_np, columns=feature_names)

# Step 9: Verify
print(fused_df_named.shape)
print(fused_df_named.head())
print(fused_df_named.tail())

(320, 33)
     feat_0    feat_1    feat_2    feat_3    feat_4    feat_5    feat_6  \
0  0.300422 -0.183858  0.175919 -0.131233  0.132250 -0.039227 -0.148647   
1 -0.267703 -0.145686 -0.220767 -0.054355 -0.197968 -0.222857 -0.262906   
2  0.226495 -0.115842  0.193838 -0.194644  0.084226  0.076668 -0.107244   
3  0.035509 -0.286301 -0.213003  0.024271  0.140168  0.136461 -0.203293   
4 -0.245758 -0.084956  0.256464 -0.062824 -0.258000 -0.115017  0.256645   

     feat_7    feat_8    feat_9  ...   feat_23   feat_24   feat_25   feat_26  \
0 -0.185258 -0.117842 -0.235565  ...  0.188710  0.259923 -0.108803 -0.018876   
1  0.217368 -0.077192 -0.184766  ... -0.092124  0.192314 -0.110669 -0.142930   
2 -0.147777 -0.228855 -0.157884  ...  0.027145  0.226666 -0.070703 -0.170178   
3  0.085391  0.288119 -0.067413  ...  0.082788  0.338707  0.086358  0.215054   
4 -0.118250  0.201528 -0.165580  ...  0.233390 -0.063842 -0.189807  0.224370   

    feat_27   feat_28   feat_29   feat_30   feat_31  PD_La

In [8]:
# Make a copy to be safe (optional)
fused_df_named_int = fused_df_named.copy()

# Convert only the last column to int
fused_df_named_int['PD_Label_Combined'] = fused_df_named_int['PD_Label_Combined'].astype(int)

# Verify
print(fused_df_named_int.dtypes)
print(fused_df_named_int.head())

feat_0               float32
feat_1               float32
feat_2               float32
feat_3               float32
feat_4               float32
feat_5               float32
feat_6               float32
feat_7               float32
feat_8               float32
feat_9               float32
feat_10              float32
feat_11              float32
feat_12              float32
feat_13              float32
feat_14              float32
feat_15              float32
feat_16              float32
feat_17              float32
feat_18              float32
feat_19              float32
feat_20              float32
feat_21              float32
feat_22              float32
feat_23              float32
feat_24              float32
feat_25              float32
feat_26              float32
feat_27              float32
feat_28              float32
feat_29              float32
feat_30              float32
feat_31              float32
PD_Label_Combined      int64
dtype: object
     feat_0    feat_1    feat

In [9]:

# ============================================================
# IMPROVED ENSEMBLE CLASSIFICATION WITH 95% CONFIDENCE INTERVALS
# Stratified 10-Fold CV + Independent Test Set
#
# Models:
#   1. XGBoost
#   2. Random Forest
#   3. HistGradientBoosting
#   4. Weighted Soft Voting
#   5. Stacking
#
# Metrics with 95% CI:
# Accuracy, Precision, Sensitivity, Specificity, F1, AUC
# ============================================================

!pip install -q xgboost scikit-learn statsmodels pandas numpy

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold
)

from sklearn.impute import SimpleImputer
from sklearn.base import clone

from sklearn.ensemble import (
    RandomForestClassifier,
    HistGradientBoostingClassifier,
    VotingClassifier,
    StackingClassifier
)

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    confusion_matrix
)

from statsmodels.stats.proportion import proportion_confint
from xgboost import XGBClassifier


# ============================================================
# STEP 1: PREPARE DATA
# ============================================================

feature_cols = [f"feat_{i}" for i in range(32)]

X = fused_df_named_int[feature_cols].copy()
y = fused_df_named_int["PD_Label_Combined"].copy()

X = X.apply(pd.to_numeric, errors="coerce")
X = X.replace([np.inf, -np.inf], np.nan)
y = pd.to_numeric(y, errors="coerce")

valid = y.notna()

X = X.loc[valid].reset_index(drop=True)
y = y.loc[valid].astype(int).reset_index(drop=True)

if set(y.unique()) != {0, 1}:
    raise ValueError("Labels must be encoded as 0 and 1.")

if X.shape[1] != 32:
    raise ValueError("Expected exactly 32 embedding features.")

if y.value_counts().min() < 10:
    raise ValueError(
        "Each class needs at least 10 samples for stratified "
        "10-fold cross-validation."
    )

print("X shape:", X.shape)
print("\nClass distribution:")
print(y.value_counts().sort_index())


# ============================================================
# STEP 2: TRAIN-TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.10,
    stratify=y,
    random_state=42
)

print("\nTraining shape:", X_train.shape)
print("Test shape:", X_test.shape)


# ============================================================
# STEP 3: MODEL FACTORY
# ============================================================

def create_models(y_fit):

    n_negative = int((y_fit == 0).sum())
    n_positive = int((y_fit == 1).sum())

    scale_pos_weight = n_negative / max(n_positive, 1)

    xgb = XGBClassifier(
        n_estimators=2500,
        max_depth=3,
        learning_rate=0.03,
        min_child_weight=3,
        subsample=0.85,
        colsample_bytree=0.85,
        gamma=0.1,
        reg_alpha=0.1,
        reg_lambda=5.0,
        scale_pos_weight=scale_pos_weight,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )

    rf = RandomForestClassifier(
        n_estimators=2500,
        max_depth=None,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features="sqrt",
        class_weight="balanced",
        bootstrap=True,
        random_state=42,
        n_jobs=-1
    )

    hgb = HistGradientBoostingClassifier(
        max_iter=2500,
        learning_rate=0.04,
        max_leaf_nodes=15,
        max_depth=5,
        min_samples_leaf=10,
        l2_regularization=2.0,
        class_weight="balanced",
        early_stopping=False,
        random_state=42
    )

    voting = VotingClassifier(
        estimators=[
            ("xgb", clone(xgb)),
            ("rf", clone(rf)),
            ("hgb", clone(hgb))
        ],
        voting="soft",
        weights=[2, 1, 1],
        n_jobs=1
    )

    stacking = StackingClassifier(
        estimators=[
            ("xgb", clone(xgb)),
            ("rf", clone(rf)),
            ("hgb", clone(hgb))
        ],
        final_estimator=LogisticRegression(
            C=0.5,
            max_iter=2000,
            class_weight="balanced"
        ),
        stack_method="predict_proba",
        passthrough=False,
        cv=StratifiedKFold(
            n_splits=5,
            shuffle=True,
            random_state=123
        ),
        n_jobs=1
    )

    return {
        "XGBoost": xgb,
        "Random Forest": rf,
        "HistGradientBoosting": hgb,
        "Voting Fusion": voting,
        "Stacking Fusion": stacking
    }


# ============================================================
# STEP 4: METRICS + 95% CONFIDENCE INTERVALS
# ============================================================

def calculate_metrics_with_ci(
    y_true,
    y_pred,
    y_prob,
    n_boot=2000,
    random_state=42
):

    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    n = len(y_true)

    # ---------------- Point estimates ----------------

    accuracy = accuracy_score(y_true, y_pred)

    precision = precision_score(
        y_true, y_pred, zero_division=0
    )

    sensitivity = recall_score(
        y_true, y_pred, zero_division=0
    )

    specificity = (
        tn / (tn + fp) if (tn + fp) > 0 else np.nan
    )

    f1 = f1_score(
        y_true, y_pred, zero_division=0
    )

    auc = roc_auc_score(y_true, y_prob)

    results = {
        "Accuracy": accuracy,
        "Precision": precision,
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "F1": f1,
        "AUC": auc,
        "Balanced Accuracy": balanced_accuracy_score(
            y_true, y_pred
        ),
        "MCC": matthews_corrcoef(y_true, y_pred),
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    }

    # ---------------- Wilson 95% CIs ----------------

    proportions = {
        "Accuracy": (tp + tn, n),
        "Precision": (tp, tp + fp),
        "Sensitivity": (tp, tp + fn),
        "Specificity": (tn, tn + fp)
    }

    for metric, (count, denominator) in proportions.items():

        if denominator > 0:
            low, high = proportion_confint(
                count=count,
                nobs=denominator,
                alpha=0.05,
                method="wilson"
            )
        else:
            low, high = np.nan, np.nan

        results[f"{metric} CI Low"] = low
        results[f"{metric} CI High"] = high

    # ---------------- Bootstrap CIs for F1 and AUC ----------------
    # Stratified resampling preserves both class counts.
    # Predictions and probabilities remain paired with labels.

    rng = np.random.default_rng(random_state)

    class_indices = [
        np.flatnonzero(y_true == 0),
        np.flatnonzero(y_true == 1)
    ]

    bootstrap_f1 = []
    bootstrap_auc = []

    for _ in range(n_boot):

        sampled_indices = np.concatenate([
            rng.choice(
                indices,
                size=len(indices),
                replace=True
            )
            for indices in class_indices
        ])

        yt = y_true[sampled_indices]
        yp = y_pred[sampled_indices]
        pr = y_prob[sampled_indices]

        bootstrap_f1.append(
            f1_score(yt, yp, zero_division=0)
        )

        bootstrap_auc.append(
            roc_auc_score(yt, pr)
        )

    for metric, values in [
        ("F1", bootstrap_f1),
        ("AUC", bootstrap_auc)
    ]:

        low, high = np.percentile(
            values,
            [2.5, 97.5]
        )

        results[f"{metric} CI Low"] = low
        results[f"{metric} CI High"] = high

    return results


# ============================================================
# STEP 5: PREPROCESSING
# ============================================================

def make_preprocessor():
    return SimpleImputer(strategy="median")


# ============================================================
# STEP 6: STRATIFIED 10-FOLD CROSS-VALIDATION
# ============================================================

skf = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

model_names = [
    "XGBoost",
    "Random Forest",
    "HistGradientBoosting",
    "Voting Fusion",
    "Stacking Fusion"
]

cv_records = []
cv_summary_records = []
test_records = []
confusion_matrices = {}

metrics_to_report = [
    "Accuracy",
    "Precision",
    "Sensitivity",
    "Specificity",
    "F1",
    "AUC"
]


for model_name in model_names:

    print("\n" + "=" * 85)
    print("EVALUATING:", model_name)
    print("=" * 85)

    fold_metrics = []

    for fold, (train_idx, val_idx) in enumerate(
        skf.split(X_train, y_train),
        start=1
    ):

        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]

        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        # Fit imputation only on this fold's training partition.
        imputer = make_preprocessor()

        X_tr_imp = imputer.fit_transform(X_tr)
        X_val_imp = imputer.transform(X_val)

        models = create_models(y_tr)
        model = models[model_name]

        model.fit(X_tr_imp, y_tr)

        val_pred = model.predict(X_val_imp)
        val_prob = model.predict_proba(X_val_imp)[:, 1]

        metrics = calculate_metrics_with_ci(
            y_val,
            val_pred,
            val_prob,
            n_boot=2000,
            random_state=1000 + fold
        )

        metrics["Model"] = model_name
        metrics["Fold"] = fold

        fold_metrics.append(metrics)
        cv_records.append(metrics.copy())

        # Print all six metrics and their CIs for each fold.
        print(f"\nFold {fold:02d}/10")

        for metric in metrics_to_report:

            print(
                f"{metric:15s}: {metrics[metric]:.4f} "
                f"(95% CI: "
                f"{metrics[metric + ' CI Low']:.4f}-"
                f"{metrics[metric + ' CI High']:.4f})"
            )

    # ========================================================
    # STEP 7: CV MEAN AND STANDARD DEVIATION
    # ========================================================

    fold_df = pd.DataFrame(fold_metrics)

    print(f"\n{model_name} - 10-Fold CV Summary")
    print("-" * 75)

    summary = {"Model": model_name}

    for metric in metrics_to_report:

        mean_val = fold_df[metric].mean()
        std_val = fold_df[metric].std(ddof=1)

        # Average lower and upper fold CI endpoints.
        mean_ci_low = fold_df[
            f"{metric} CI Low"
        ].mean()

        mean_ci_high = fold_df[
            f"{metric} CI High"
        ].mean()

        summary[f"CV {metric} Mean"] = mean_val
        summary[f"CV {metric} SD"] = std_val
        summary[f"Mean Fold {metric} CI Low"] = mean_ci_low
        summary[f"Mean Fold {metric} CI High"] = mean_ci_high

        print(
            f"{metric:15s}: {mean_val:.4f} +/- {std_val:.4f} "
            f"| Mean fold CI: "
            f"{mean_ci_low:.4f}-{mean_ci_high:.4f}"
        )

    for metric in ["Balanced Accuracy", "MCC"]:

        summary[f"CV {metric} Mean"] = fold_df[metric].mean()
        summary[f"CV {metric} SD"] = fold_df[metric].std(ddof=1)

    cv_summary_records.append(summary)

    # ========================================================
    # STEP 8: FINAL MODEL AND INDEPENDENT TEST SET
    # ========================================================

    final_imputer = make_preprocessor()

    X_train_imp = final_imputer.fit_transform(X_train)
    X_test_imp = final_imputer.transform(X_test)

    final_models = create_models(y_train)
    final_model = final_models[model_name]

    final_model.fit(X_train_imp, y_train)

    test_pred = final_model.predict(X_test_imp)
    test_prob = final_model.predict_proba(X_test_imp)[:, 1]

    test_metrics = calculate_metrics_with_ci(
        y_test,
        test_pred,
        test_prob,
        n_boot=5000,
        random_state=2026
    )

    test_metrics["Model"] = model_name
    test_records.append(test_metrics.copy())

    confusion_matrices[model_name] = confusion_matrix(
        y_test,
        test_pred,
        labels=[0, 1]
    )

    print(f"\n{model_name} - Independent Test Set")
    print("-" * 75)

    for metric in metrics_to_report:

        print(
            f"{metric:15s}: {test_metrics[metric]:.4f} "
            f"(95% CI: "
            f"{test_metrics[metric + ' CI Low']:.4f}-"
            f"{test_metrics[metric + ' CI High']:.4f})"
        )

    print(
        f"Balanced Accuracy: "
        f"{test_metrics['Balanced Accuracy']:.4f}"
    )

    print(f"MCC: {test_metrics['MCC']:.4f}")

    print("\nConfusion Matrix:")
    print(confusion_matrices[model_name])


# ============================================================
# STEP 9: INDIVIDUAL FOLD RESULTS WITH 95% CIs
# ============================================================

fold_results_df = pd.DataFrame(cv_records)

fold_columns = [
    "Model", "Fold",
    "Accuracy", "Accuracy CI Low", "Accuracy CI High",
    "Precision", "Precision CI Low", "Precision CI High",
    "Sensitivity", "Sensitivity CI Low", "Sensitivity CI High",
    "Specificity", "Specificity CI Low", "Specificity CI High",
    "F1", "F1 CI Low", "F1 CI High",
    "AUC", "AUC CI Low", "AUC CI High",
    "Balanced Accuracy", "MCC",
    "TN", "FP", "FN", "TP"
]

fold_results_df = fold_results_df[fold_columns]

print("\n" + "=" * 100)
print("ALL MODELS: INDIVIDUAL FOLD RESULTS WITH 95% CIs")
print("=" * 100)

display(
    fold_results_df.sort_values(
        ["Model", "Fold"]
    ).round(4)
)


# ============================================================
# STEP 10: CROSS-VALIDATION SUMMARY
# ============================================================

cv_summary_df = pd.DataFrame(cv_summary_records)

print("\n" + "=" * 100)
print("CROSS-VALIDATION MEAN, SD AND MEAN FOLD CIs")
print("=" * 100)

display(cv_summary_df.round(4))


# ============================================================
# STEP 11: INDEPENDENT TEST RESULTS WITH 95% CIs
# ============================================================

test_results_df = pd.DataFrame(
    test_records
).set_index("Model")

test_columns = [
    "Accuracy", "Accuracy CI Low", "Accuracy CI High",
    "Precision", "Precision CI Low", "Precision CI High",
    "Sensitivity", "Sensitivity CI Low", "Sensitivity CI High",
    "Specificity", "Specificity CI Low", "Specificity CI High",
    "F1", "F1 CI Low", "F1 CI High",
    "AUC", "AUC CI Low", "AUC CI High",
    "Balanced Accuracy", "MCC",
    "TN", "FP", "FN", "TP"
]

test_results_df = test_results_df[test_columns]

print("\n" + "=" * 100)
print("INDEPENDENT TEST SET COMPARISON WITH 95% CIs")
print("=" * 100)

display(
    test_results_df.sort_values(
        by="F1",
        ascending=False
    ).round(4)
)


# ============================================================
# STEP 12: VOTING ENSEMBLE RESULTS
# ============================================================

print("\n" + "=" * 100)
print("WEIGHTED VOTING ENSEMBLE RESULTS")
print("=" * 100)

print("Voting weights: XGBoost=2, Random Forest=1, HGB=1")

print("\nVoting fold results:")
display(
    fold_results_df[
        fold_results_df["Model"] == "Voting Fusion"
    ].round(4)
)

print("\nVoting cross-validation summary:")
display(
    cv_summary_df[
        cv_summary_df["Model"] == "Voting Fusion"
    ].round(4)
)

print("\nVoting independent test results:")
display(
    test_results_df.loc[["Voting Fusion"]].round(4)
)

X shape: (320, 32)

Class distribution:
PD_Label_Combined
0    131
1    189
Name: count, dtype: int64

Training shape: (288, 32)
Test shape: (32, 32)

EVALUATING: XGBoost

Fold 01/10
Accuracy       : 0.6897 (95% CI: 0.5077-0.8272)
Precision      : 0.7857 (95% CI: 0.5241-0.9243)
Sensitivity    : 0.6471 (95% CI: 0.4130-0.8269)
Specificity    : 0.7500 (95% CI: 0.4677-0.9111)
F1             : 0.7097 (95% CI: 0.5000-0.8667)
AUC            : 0.6961 (95% CI: 0.4755-0.8725)

Fold 02/10
Accuracy       : 0.5862 (95% CI: 0.4074-0.7449)
Precision      : 0.6471 (95% CI: 0.4130-0.8269)
Sensitivity    : 0.6471 (95% CI: 0.4130-0.8269)
Specificity    : 0.5000 (95% CI: 0.2538-0.7462)
F1             : 0.6471 (95% CI: 0.4571-0.8108)
AUC            : 0.6912 (95% CI: 0.4705-0.8725)

Fold 03/10
Accuracy       : 0.7586 (95% CI: 0.5789-0.8778)
Precision      : 0.9167 (95% CI: 0.6461-0.9851)
Sensitivity    : 0.6471 (95% CI: 0.4130-0.8269)
Specificity    : 0.9167 (95% CI: 0.6461-0.9851)
F1             : 0.7586 (

,Model,Fold,Accuracy,Accuracy CI Low,Accuracy CI High,Precision,Precision CI Low,Precision CI High,Sensitivity,Sensitivity CI Low,...,F1 CI High,AUC,AUC CI Low,AUC CI High,Balanced Accuracy,MCC,TN,FP,FN,TP
20,HistGradientBoosting,1,0.6207,0.4400,0.7731,0.7143,0.4535,0.8828,0.5882,0.3601,...,0.8125,0.7010,0.4853,0.8873,0.6275,0.2512,8,4,7,10
21,HistGradientBoosting,2,0.6552,0.4735,0.8006,0.7059,0.4687,0.8672,0.7059,0.4687,...,0.8571,0.6667,0.4510,0.8627,0.6446,0.2892,7,5,5,12
22,HistGradientBoosting,3,0.7241,0.5428,0.8530,0.8462,0.5777,0.9567,0.6471,0.4130,...,0.8889,0.7990,0.6176,0.9461,0.7402,0.4758,10,2,6,11
23,HistGradientBoosting,4,0.6552,0.4735,0.8006,0.7059,0.4687,0.8672,0.7059,0.4687,...,0.8571,0.7843,0.5882,0.9314,0.6446,0.2892,7,5,5,12
24,HistGradientBoosting,5,0.6897,0.5077,0.8272,0.6667,0.4671,0.8203,0.9412,0.7302,...,0.8718,0.7108,0.5098,0.8873,0.6373,0.3579,4,8,1,16
25,HistGradientBoosting,6,0.6207,0.4400,0.7731,0.6667,0.4375,0.8372,0.7059,0.4687,...,0.8333,0.6275,0.4067,0.8284,0.6029,0.2090,6,6,5,12
26,HistGradientBoosting,7,0.5862,0.4074,0.7449,0.6471,0.4130,0.8269,0.6471,0.4130,...,0.8000,0.6078,0.3920,0.8186,0.5735,0.1471,6,6,6,11
27,HistGradientBoosting,8,0.5517,0.3755,0.7159,0.6250,0.3864,0.8152,0.5882,0.3601,...,0.7647,0.6225,0.3971,0.8186,0.5441,0.0874,6,6,7,10
28,HistGradientBoosting,9,0.7143,0.5294,0.8475,0.8000,0.5481,0.9295,0.7059,0.4687,...,0.8889,0.7701,0.5508,0.9519,0.7166,0.4242,8,3,5,12
29,HistGradientBoosting,10,0.7500,0.5664,0.8732,0.7778,0.5479,0.9100,0.8235,0.5897,...,0.9143,0.7487,0.5401,0.9092,0.7299,0.4688,7,4,3,14



CROSS-VALIDATION MEAN, SD AND MEAN FOLD CIs


,Model,CV Accuracy Mean,CV Accuracy SD,Mean Fold Accuracy CI Low,Mean Fold Accuracy CI High,CV Precision Mean,CV Precision SD,Mean Fold Precision CI Low,Mean Fold Precision CI High,CV Sensitivity Mean,...,Mean Fold F1 CI Low,Mean Fold F1 CI High,CV AUC Mean,CV AUC SD,Mean Fold AUC CI Low,Mean Fold AUC CI High,CV Balanced Accuracy Mean,CV Balanced Accuracy SD,CV MCC Mean,CV MCC SD
0,XGBoost,0.6183,0.0848,0.4393,0.7695,0.6967,0.1082,0.4556,0.8572,0.6529,...,0.4813,0.8222,0.6952,0.0677,0.4839,0.8766,0.6109,0.0961,0.2238,0.1889
1,Random Forest,0.6429,0.0691,0.4623,0.7897,0.7044,0.0691,0.4651,0.8651,0.6882,...,0.5120,0.8376,0.7166,0.0643,0.5123,0.8885,0.6328,0.0716,0.2689,0.1413
2,HistGradientBoosting,0.6568,0.0635,0.4756,0.8009,0.7155,0.0714,0.4768,0.8713,0.7059,...,0.5318,0.8489,0.7038,0.0707,0.4939,0.8841,0.6461,0.0657,0.3000,0.1323
3,Voting Fusion,0.6325,0.0776,0.4526,0.7811,0.7047,0.1011,0.4652,0.8619,0.6765,...,0.5032,0.8295,0.7169,0.0696,0.5114,0.8898,0.6231,0.0899,0.2468,0.1763
4,Stacking Fusion,0.6357,0.0596,0.4550,0.7845,0.7236,0.0882,0.4731,0.8792,0.6412,...,0.4818,0.8295,0.7135,0.0663,0.5082,0.8858,0.6346,0.0723,0.2680,0.1414



INDEPENDENT TEST SET COMPARISON WITH 95% CIs


...

,Model,Fold,Accuracy,Accuracy CI Low,Accuracy CI High,Precision,Precision CI Low,Precision CI High,Sensitivity,Sensitivity CI Low,...,F1 CI High,AUC,AUC CI Low,AUC CI High,Balanced Accuracy,MCC,TN,FP,FN,TP
30,Voting Fusion,1,0.6207,0.4400,0.7731,0.7143,0.4535,0.8828,0.5882,0.3601,...,0.8125,0.7255,0.5147,0.8922,0.6275,0.2512,8,4,7,10
31,Voting Fusion,2,0.5862,0.4074,0.7449,0.6471,0.4130,0.8269,0.6471,0.4130,...,0.8000,0.6961,0.4804,0.8725,0.5735,0.1471,6,6,6,11
32,Voting Fusion,3,0.7586,0.5789,0.8778,0.9167,0.6461,0.9851,0.6471,0.4130,...,0.9091,0.8529,0.6912,0.9804,0.7819,0.5637,11,1,6,11
33,Voting Fusion,4,0.6207,0.4400,0.7731,0.6875,0.4440,0.8584,0.6471,0.4130,...,0.8235,0.7402,0.5343,0.9020,0.6152,0.2282,7,5,6,11
34,Voting Fusion,5,0.5862,0.4074,0.7449,0.6000,0.4074,0.7660,0.8824,0.6566,...,0.8000,0.7255,0.5245,0.8971,0.5245,0.0700,2,10,2,15
35,Voting Fusion,6,0.5517,0.3755,0.7159,0.6111,0.3862,0.7969,0.6471,0.4130,...,0.7879,0.6667,0.4558,0.8578,0.5319,0.0647,5,7,6,11
36,Voting Fusion,7,0.6552,0.4735,0.8006,0.7059,0.4687,0.8672,0.7059,0.4687,...,0.8571,0.6667,0.4608,0.8627,0.6446,0.2892,7,5,5,12
37,Voting Fusion,8,0.5172,0.3443,0.6861,0.6000,0.3575,0.8018,0.5294,0.3096,...,0.7273,0.5980,0.3724,0.7990,0.5147,0.0290,6,6,8,9
38,Voting Fusion,9,0.7143,0.5294,0.8475,0.8000,0.5481,0.9295,0.7059,0.4687,...,0.8889,0.7861,0.5829,0.9465,0.7166,0.4242,8,3,5,12
39,Voting Fusion,10,0.7143,0.5294,0.8475,0.7647,0.5274,0.9044,0.7647,0.5274,...,0.8889,0.7112,0.4972,0.8877,0.7005,0.4011,7,4,4,13



Voting cross-validation summary:


,Model,CV Accuracy Mean,CV Accuracy SD,Mean Fold Accuracy CI Low,Mean Fold Accuracy CI High,CV Precision Mean,CV Precision SD,Mean Fold Precision CI Low,Mean Fold Precision CI High,CV Sensitivity Mean,...,Mean Fold F1 CI Low,Mean Fold F1 CI High,CV AUC Mean,CV AUC SD,Mean Fold AUC CI Low,Mean Fold AUC CI High,CV Balanced Accuracy Mean,CV Balanced Accuracy SD,CV MCC Mean,CV MCC SD
3,Voting Fusion,0.6325,0.0776,0.4526,0.7811,0.7047,0.1011,0.4652,0.8619,0.6765,...,0.5032,0.8295,0.7169,0.0696,0.5114,0.8898,0.6231,0.0899,0.2468,0.1763



Voting independent test results:


,Accuracy,Accuracy CI Low,Accuracy CI High,Precision,Precision CI Low,Precision CI High,Sensitivity,Sensitivity CI Low,Sensitivity CI High,Specificity,...,F1 CI High,AUC,AUC CI Low,AUC CI High,Balanced Accuracy,MCC,TN,FP,FN,TP
Model,,,,,,,,,,,,,,,,,,,,,
Voting Fusion,0.5312,0.3645,0.6913,0.5909,0.3873,0.7674,0.6842,0.4601,0.8464,0.3077,...,0.7619,0.6721,0.4696,0.8462,0.496,-0.0086,4,9,6,13
